# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library.

### Dataset Source

The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```

The dataset contains structured tabulations of clinical features, hormone levels, adrenal imaging, and genetic variants for three female patients diagnosed with non-classical 21-hydroxylase deficiency-associated infertility.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. This section initializes the dataset object and prints basic metadata information.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Accessing metadata as object
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")

## 2. Data Overview

Review available record sets, fields, columns, and their `@id`s as defined in the Croissant schema. Below, we list all record sets and their fields with `@id` references.

In [ ]:
# Explore record sets and their structure using @id

record_sets = dataset.record_sets
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"RecordSet name: {rs['name']}")
    print("Fields:")
    for field in rs.get('fields', []):
        print(f"  Field @id: {field['@id']} | name: {field['name']} | dataType: {field.get('dataType', 'Unknown')}")
    print("Columns:")
    for col in rs.get('columns', []):
        print(f"  Column @id: {col['@id']} | name: {col['name']} | dataType: {col.get('dataType', 'Unknown')}")
    print("-")

## 3. Data Extraction

Load data from each record set into separate DataFrames for analysis.

Below, we use the record set and field `@id`s as discovered above:

In [ ]:
# Extract data from all record sets using @id

dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet @id: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(), "\n")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. All entities are referenced by their `@id`.

For example, let's select a numeric field from the first record set and perform filtering and normalization.

In [ ]:
# Example EDA for the first record set
import numpy as np

# Using the first record set
first_rs_id = record_set_ids[0]
df = dataframes[first_rs_id]

# Try to get a numeric field @id from the columns
numeric_columns = []
for rs in dataset.record_sets:
    if rs['@id'] == first_rs_id:
        numeric_columns = [col['@id'] for col in rs.get('columns', []) if col.get('dataType', '').lower() in ['integer', 'float', 'number']]
        break

if numeric_columns:
    numeric_field_id = numeric_columns[0]

    # Set an example threshold
    threshold = 10

    # Filtering
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalizing
    mean_val = filtered_df[numeric_field_id].mean()
    std_val = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_val) / std_val
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Check for a group field
    group_field_id = None
    potential_group_fields = [col['@id'] for col in rs.get('columns', []) if col.get('dataType', '').lower() not in ['integer', 'float', 'number']]
    for pf in potential_group_fields:
        if pf in df.columns:
            group_field_id = pf
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric fields found in the first record set for EDA.")

## 5. Visualization

Visualize the distribution of a numeric field or relationship between fields using matplotlib and seaborn.

In [ ]:
# Visualization example
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in RecordSet {first_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    if group_field_id:
        plt.figure(figsize=(6,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric columns found for visualization.")

## 6. Conclusion

In this notebook, we:
- Loaded the FAIR^2 dataset using its Croissant schema URL and explored its metadata.
- Reviewed available record sets and fields using their `@id`.
- Extracted data into pandas DataFrames, referencing entities by `@id`, and performed filtering and normalization on numeric fields.
- Visualized distributions and relationships within the dataset.

This structured approach enables transparent, reproducible FAIR data exploration. For further analysis, refer to `mlcroissant` documentation and extend these notebooks for statistical modeling and clinical correlation studies.